# NeuroScan AI · retrain VGG16 on head-cropped slices

Grad-CAM showed the first VGG16 model classified **gliomas from image borders and background**, not the tumor
(0% of its strong attention on head tissue). This notebook removes that shortcut: every slice is cropped to the
head (largest bright region, borders and background blanked) before training and evaluation.
The backend applies the identical crop automatically when it sees `"preprocess": "crop_head"` in the metrics file.

Produces `vgg16_classifier.pt` and `vgg16_classifier.metrics.json` (replace the old ones).

**Colab:** Runtime → Change runtime type → T4 GPU → Run all → allow Google Drive. Files also land in My Drive → NeuroScanAI → weights_cropped.
**Kaggle:** Import notebook → Accelerator GPU T4 x2, Internet On → Run All → download `neuroscan_vgg16_cropped.zip` from Output.

About 15 minutes on a T4. Research and education only.

In [ ]:
# Settings for the head-cropped VGG16 run. Works on Colab and Kaggle.
import os, sys, json, time, random, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
SMOKE = os.environ.get("NEUROSCAN_SMOKE") == "1"   # tiny run used to test this notebook on a CPU

WORK = Path(os.environ.get("NEUROSCAN_WORK", "/kaggle/working" if IN_KAGGLE else "/content"))
CLS_RAW = WORK / "nickparvar"
OUT = WORK / "neuroscan_weights"

# Save every checkpoint straight to Google Drive, so a dropped Colab session loses nothing.
SAVE_TO_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/NeuroScanAI"

CLS_EPOCHS = 1 if SMOKE else 12
CLS_PER_CLASS_LIMIT = 8 if SMOKE else None   # None = use every image
SEED = 42

if IN_COLAB or IN_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)
    if IN_COLAB and SAVE_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")   # a Google sign-in popup appears: allow access
        OUT = Path(DRIVE_DIR) / "weights_cropped"

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
from torchvision.models import VGG16_Weights, vgg16

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
OUT.mkdir(parents=True, exist_ok=True)
print("saving to:", OUT)
print("device:", DEVICE, torch.cuda.get_device_name(0) if AMP else "", "| smoke test" if SMOKE else "")
if not AMP and not SMOKE:
    print("WARNING: no GPU. Runtime → Change runtime type → T4 GPU, then run again.")

## Model definition

Copied from `backend/app/models/vgg16_classifier.py`.

In [ ]:
CLASS_NAMES = ("glioma", "meningioma", "no_tumor", "pituitary")
SEG_CLASSES = ("background", "necrotic_core", "edema", "enhancing")
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]


def build_vgg16_classifier(num_classes=len(CLASS_NAMES), pretrained=True):
    model = vgg16(weights=VGG16_Weights.IMAGENET1K_V1 if pretrained else None)
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
    return model


print("model defined")

## 1 · Head crop

In [ ]:
# Head crop: identical to backend/app/preprocess.py. Keep both in sync.
import numpy as np
from scipy import ndimage


def crop_to_head(gray: np.ndarray, margin: float = 0.04) -> np.ndarray:
    """Square crop around the largest bright connected region (the head), zero padded."""
    thr = max(15.0, float(np.percentile(gray, 30)) * 0.5)
    mask = ndimage.binary_opening(gray > thr, iterations=2)
    labels, n = ndimage.label(mask)
    if n == 0:
        return gray
    sizes = ndimage.sum(mask, labels, index=range(1, n + 1))
    head = labels == (int(np.argmax(sizes)) + 1)
    if head.sum() < 0.02 * gray.size:  # nothing head-like found; leave the image alone
        return gray

    ys, xs = np.where(head)
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    pad = int(margin * max(y1 - y0, x1 - x0))
    y0, x0 = max(0, y0 - pad), max(0, x0 - pad)
    y1, x1 = min(gray.shape[0], y1 + pad), min(gray.shape[1], x1 + pad)

    # Keep only the head component so text, rulers or border lines inside the box vanish.
    region = np.where(ndimage.binary_dilation(head, iterations=3), gray, 0)[y0:y1, x0:x1]
    h, w = region.shape
    side = max(h, w)
    out = np.zeros((side, side), dtype=gray.dtype)
    out[(side - h) // 2:(side - h) // 2 + h, (side - w) // 2:(side - w) // 2 + w] = region
    return out


class CropToHead:
    """torchvision-style transform: PIL image in, head-cropped grayscale PIL image out."""
    def __call__(self, img):
        return Image.fromarray(crop_to_head(np.asarray(img.convert("L"))))

## 2 · Classifier data (7,200 MRI slices, 4 classes)

In [ ]:
from huggingface_hub import list_repo_files, snapshot_download

if not any(CLS_RAW.glob("Training/*/*.jpg")):
    t = time.time()
    snapshot_download("BJyotibrat/Masoud-Nickparvar-Brain-Tumor-MRI-Dataset", repo_type="dataset",
                      local_dir=CLS_RAW, allow_patterns=["Training/*", "Testing/*"], max_workers=16)
    print(f"downloaded in {time.time() - t:.0f}s")

def tf(train):
    steps = [CropToHead(), transforms.Grayscale(3), transforms.Resize((224, 224))]
    if train:
        steps += [transforms.RandomHorizontalFlip(), transforms.RandomRotation(10),
                  transforms.ColorJitter(brightness=0.15, contrast=0.15)]
    return transforms.Compose(steps + [transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

train_full = datasets.ImageFolder(CLS_RAW / "Training", transform=tf(True))
train_eval = datasets.ImageFolder(CLS_RAW / "Training", transform=tf(False))
test_ds = datasets.ImageFolder(CLS_RAW / "Testing", transform=tf(False))
# Folder "notumor" sorts into the same slot as our "no_tumor".
assert [c.replace("notumor", "no_tumor") for c in train_full.classes] == list(CLASS_NAMES), train_full.classes

def limit(ds, n):
    if n is None:
        return list(range(len(ds)))
    seen, keep = {}, []
    for i, (_, y) in enumerate(ds.samples):
        if seen.get(y, 0) < n:
            keep.append(i); seen[y] = seen.get(y, 0) + 1
    return keep

# 10% of Training is held out for model selection; Testing is only used for the final report.
idx = limit(train_full, CLS_PER_CLASS_LIMIT)
random.Random(SEED).shuffle(idx)
n_val = max(4, len(idx) // 10)
val_idx, tr_idx = idx[:n_val], idx[n_val:]
test_idx = limit(test_ds, CLS_PER_CLASS_LIMIT)

workers = 2 if (IN_COLAB or IN_KAGGLE) else 0
cls_train = DataLoader(Subset(train_full, tr_idx), batch_size=32, shuffle=True, num_workers=workers, pin_memory=AMP)
cls_val = DataLoader(Subset(train_eval, val_idx), batch_size=64, num_workers=workers, pin_memory=AMP)
cls_test = DataLoader(Subset(test_ds, test_idx), batch_size=64, num_workers=workers, pin_memory=AMP)
print(f"train {len(tr_idx)} · val {len(val_idx)} · test {len(test_idx)}")

## 3 · Fine-tune VGG16

In [ ]:
def run_cls_epoch(model, loader, opt=None, scaler=None):
    train = opt is not None
    model.train(train)
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)
    loss_sum = correct = seen = 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(DEVICE.type, enabled=AMP):
                logits = model(x)
                loss = crit(logits, y)
            if train:
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            loss_sum += loss.item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()
            seen += len(y)
    return loss_sum / seen, correct / seen

clf = build_vgg16_classifier().to(DEVICE)
# Lower learning rate for the pretrained conv layers, higher for the classifier head.
opt = torch.optim.AdamW([
    {"params": clf.features.parameters(), "lr": 2e-5},
    {"params": clf.classifier.parameters(), "lr": 2e-4},
], weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CLS_EPOCHS)
scaler = torch.amp.GradScaler(enabled=AMP)

best_acc, cls_path = -1.0, OUT / "vgg16_classifier.pt"
for epoch in range(1, CLS_EPOCHS + 1):
    t = time.time()
    tr_loss, tr_acc = run_cls_epoch(clf, cls_train, opt, scaler)
    va_loss, va_acc = run_cls_epoch(clf, cls_val)
    sched.step()
    flag = ""
    if va_acc > best_acc:
        best_acc = va_acc
        torch.save({k: v.detach().cpu() for k, v in clf.state_dict().items()}, cls_path)
        flag = "  ← saved"
    print(f"epoch {epoch:02d}  train {tr_loss:.3f}/{tr_acc:.3f}  val {va_loss:.3f}/{va_acc:.3f}  {time.time() - t:.0f}s{flag}")

In [ ]:
# Final evaluation on the untouched Testing split, with the best checkpoint.
clf.load_state_dict(torch.load(cls_path, map_location=DEVICE, weights_only=True))
clf.eval()
n = len(CLASS_NAMES)
cm = [[0] * n for _ in range(n)]
with torch.inference_mode():
    for x, y in cls_test:
        pred = clf(x.to(DEVICE)).argmax(1).cpu()
        for t_, p_ in zip(y.tolist(), pred.tolist()):
            cm[t_][p_] += 1

per_class = {}
for i, name in enumerate(CLASS_NAMES):
    tp = cm[i][i]; fn = sum(cm[i]) - tp; fp = sum(cm[r][i] for r in range(n)) - tp
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    per_class[name] = {"precision": round(prec, 4), "recall": round(rec, 4),
                       "f1": round(2 * prec * rec / (prec + rec), 4) if prec + rec else 0.0, "support": sum(cm[i])}
total = sum(map(sum, cm))
cls_metrics = {
    "accuracy": round(sum(cm[i][i] for i in range(n)) / total, 4),
    "classes": list(CLASS_NAMES),
    "confusion_matrix": cm,
    "per_class": per_class,
    "val_images": total,
    "mode": "full fine-tune, head-cropped (GPU)",
    "preprocess": "crop_head",
    "epochs": CLS_EPOCHS,
    "best_val_acc_during_training": round(best_acc, 4),
}
(OUT / "vgg16_classifier.metrics.json").write_text(json.dumps(cls_metrics, indent=2))
print(f"TEST ACCURACY {cls_metrics['accuracy']:.4f}")
for k, v in per_class.items():
    print(f"  {k:<11} P {v['precision']:.3f}  R {v['recall']:.3f}  F1 {v['f1']:.3f}")

## 4 · Check and download

In [ ]:
import shutil

# Reload the way the backend does (strict key match, weights_only) and preview the crop.
m = build_vgg16_classifier(pretrained=False)
m.load_state_dict(torch.load(OUT / "vgg16_classifier.pt", map_location="cpu", weights_only=True))
with torch.inference_mode():
    assert m.eval()(torch.zeros(1, 3, 224, 224)).shape == (1, 4)

pack = WORK / "neuroscan_vgg16_cropped"
pack.mkdir(exist_ok=True)
for name in ("vgg16_classifier.pt", "vgg16_classifier.metrics.json"):
    shutil.copy2(OUT / name, pack / name)
archive = shutil.make_archive(str(OUT.parent / "neuroscan_vgg16_cropped"), "zip", pack)
for f in sorted(pack.iterdir()):
    print(f"{f.name:<32} {f.stat().st_size / 1e6:7.1f} MB")
print("archive:", archive)

if IN_COLAB:
    from google.colab import files
    files.download(archive)
elif IN_KAGGLE:
    print("Download neuroscan_vgg16_cropped.zip from the Output panel (/kaggle/working).")

## Install on your computer

Replace the old classifier files in `backend/weights/` with the two new ones:

```bash
cd "Longitutdinal Brain Tumour Segmentation/backend"
unzip -o ~/Downloads/neuroscan_vgg16_cropped.zip -d weights/
```

Restart the backend. `/api/health` then reports `"classifier_preprocess": "crop_head"` and the crop is applied to every upload.